# Fraud Project Final - Báo cáo có thể truy vết

Notebook tổng hợp này được thực thi sau 01, 02 và 03 bởi `scripts/run_full_project.py`. Nó chỉ đọc các artifact của cùng lần chạy cuối để tránh vô tình chọn lại model/threshold sau khi đã mở test.

In [ ]:
from pathlib import Path
import json
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'config' / 'a1_config.json').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc dự án.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
from IPython.display import Image, display
import pandas as pd

audit = pd.read_csv(PROJECT_ROOT / 'outputs/tables/data_audit.csv')
comparison = pd.read_csv(PROJECT_ROOT / 'outputs/tables/model_comparison.csv')
top_p = pd.read_csv(PROJECT_ROOT / 'outputs/tables/top_p_metrics.csv')
evaluation = json.loads((PROJECT_ROOT / 'outputs/tables/evaluation_summary.json').read_text(encoding='utf-8'))
display(audit)
display(comparison)
display(top_p)

In [ ]:
test = evaluation['test']
primary = test['cost_threshold_metrics']
print(f"Mô hình chính: {evaluation['selected_family']}")
print(f"Test AP: {test['average_precision']:.6f} (95% bootstrap CI {test['ap_bootstrap_95_ci'][0]:.6f} - {test['ap_bootstrap_95_ci'][1]:.6f})")
print(f"ROC-AUC: {test['roc_auc']:.6f}")
print(f"Threshold chi phí: {primary['threshold']:.8f}; Precision={primary['precision']:.4f}; Recall={primary['recall']:.4f}; F1={primary['f1']:.4f}")

In [ ]:
for name in ['class_distribution.png', 'amount_by_class.png', 'time_by_class.png', 'selected_correlations.png', 'validation_pr_curve.png', 'test_pr_curve.png', 'test_confusion_matrix.png', 'top_p_performance.png', 'feature_importance.png']:
    path = PROJECT_ROOT / 'outputs/figures' / name
    assert path.is_file(), path
    display(Image(filename=str(path)))

**Giới hạn:** dữ liệu chỉ bao phủ khoảng hai ngày, V1-V28 đã ẩn danh, không có lịch sử khách hàng và giả định chi phí là học thuật. Đầu ra dùng để xếp hạng ưu tiên kiểm tra, không tự động khóa thẻ.